# Field model layer overview

This notebook explains the **field representation** (field belief model) layer of OceanBench: why it exists, the common interface, and how it fits into the larger benchmark.

## Purpose

Before implementing planners or multi-robot logic, we need a reliable way to **estimate ocean fields from sparse observations** and to compare different representations. The field-model layer provides:

- A **common API** (`FieldBeliefModel`) so that tasks and planners can use any implementation interchangeably.
- **Six implementations**: Local Linear (baseline), full GP, Sparse Online GP, Pseudo-Input GP, Spatio-Temporal GP, and GMRF.
- **Evaluation** on held-out truth (RMSE, MAE, timing) via `oceanbench-bench` and `oceanbench-tasks`.

## Common interface

All field models support:

- `fit(observations)` — cold start from an `ObservationBatch`
- `update(observations)` — incremental update (only some models support this)
- `predict(query_points)` — returns `FieldPrediction(mean, std, metadata)` with NumPy arrays
- `supports_uncertainty`, `supports_online_update` — capability flags
- `reset()` — clear state, keep config
- Config and `seed` in the constructor for reproducibility.

In [1]:
# Ensure repo root is on path
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in ["oceanbench-core", "oceanbench-models"]:
    d = ROOT / p
    if d.exists() and str(d) not in sys.path:
        sys.path.insert(0, str(d))

from oceanbench_core.types import ObservationBatch, QueryPoints
from oceanbench_models.belief.field import (
    FieldBeliefModel,
    LocalLinearFieldModel,
    GPFieldModel,
)

import numpy as np
rng = np.random.default_rng(42)
n = 60
obs = ObservationBatch(
    lats=rng.uniform(24, 36, n),
    lons=rng.uniform(-86, -74, n),
    values=20.0 + 0.1 * rng.standard_normal(n),
    variable="temp",
)
qp = QueryPoints(lats=rng.uniform(24, 36, 30), lons=rng.uniform(-86, -74, 30))

for name, model in [("Local Linear", LocalLinearFieldModel(seed=42)), ("GP", GPFieldModel(seed=42))]:
    model.fit(obs)
    pred = model.predict(qp)
    print(f"{name}: mean shape={pred.mean.shape}, std={pred.std is not None}")

Local Linear: mean shape=(30,), std=False
GP: mean shape=(30,), std=True


See `docs/field_models.md` for a table of all models and their capabilities. See `examples/compare_field_models.py` for a full comparison pipeline.